# 01 — Data Source Layer

This notebook loads the raw FOI dengue data and the PSA census reference data, then runs the
data quality checks defined in `src/validation.py`.

No transformation happens in this notebook — cleaning and standardisation begin in notebook 02.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make src/ importable when running from notebooks/
sys.path.append(str(Path.cwd().parent))

from src.validation import (
    YEARS,
    FOI_GRAND_TOTALS,
    check_grand_totals,
    check_panel_shape,
    check_sanity,
    check_age_sex,
    check_census_totals,
)

RAW = Path.cwd().parent / "data" / "01_raw"
REF = Path.cwd().parent / "data" / "02_official_reference"

print("Raw source files:")
for p in sorted(RAW.iterdir()):
    if p.is_file():
        print("  ", p.name)
print("\nOfficial reference files:")
for p in sorted(REF.iterdir()):
    if p.is_file():
        print("  ", p.name)

Raw source files:
   .gitkeep
   dengue_datasets__age_sex_2021.csv
   dengue_datasets__age_sex_2022.csv
   dengue_datasets__age_sex_2023.csv
   dengue_datasets__age_sex_2024.csv
   dengue_datasets__age_sex_2025.csv
   dengue_datasets__local_govt_level.csv
   EFOI_Dengue_Cases_in_the_National_Capital_Region_2021-2025...pdf

Official reference files:
   .gitkeep
   ncr_population_masterlist__2020.csv
   ncr_population_masterlist__2024.csv


## 1. Load the FOI dengue case counts

Source: DOH-MMCHD Freedom of Information release

In [11]:
cases = pd.read_csv(RAW / "dengue_datasets__local_govt_level.csv", thousands=",")

print(f"Shape: {cases.shape[0]} LGUs x {cases.shape[1] - 1} years")
cases

Shape: 17 LGUs x 5 years


,LGU,2021,2022,2023,2024,2025
0,Caloocan,1051,4371,2334,3262,5451
1,Las Piñas,495,1603,969,1506,2132
2,Makati,457,1967,1136,1055,1103
3,Malabon,459,2520,1299,1110,1349
4,Mandaluyong,319,952,543,946,901
5,Manila,1220,3919,2712,5511,8601
6,Marikina,330,1305,699,1450,1166
7,Muntinlupa,198,1268,383,972,980
8,Navotas,201,950,191,535,590
9,Parañaque,594,2281,1151,1697,1611


## 2. Cross-source consistency check

The sum of LGU-level case counts must equal the NCR grand total printed in the FOI release, for each year separately. 

In [12]:
results = check_grand_totals(cases)

pd.DataFrame(results).T.assign(
    match=lambda d: d["match"].map({True: "OK", False: "MISMATCH"})
)

,computed,reported,match
2021,10493,10493,OK
2022,43753,43753,OK
2023,23678,23678,OK
2024,37225,37225,OK
2025,45014,45014,OK


## 3. Panel completeness

The study covers the 17 LGUs of Metro Manila — 16 highly urbanised cities plus the municipality
of Pateros — across 5 years, giving an 85-row LGU-year panel once reshaped to long format.

This check confirms no LGU is missing and no unexpected name has been introduced during
transcription.

In [13]:
shape = check_panel_shape(cases)

for key, value in shape.items():
    print(f"{key:>20}: {value}")

           row_count: 17
       expected_rows: 17
     unexpected_lgus: []
        missing_lgus: []
    long_format_rows: 85


## 4. Sanity checks

Case counts must be non-negative, non-null integers. Extreme values are inspected rather than
rejected — dengue case counts vary widely across LGUs by design, since Quezon City has roughly
45 times the population of Pateros.

In [14]:
sanity = check_sanity(cases)

for key, value in sanity.items():
    print(f"{key:>18}: {value:,}" if isinstance(value, int) else f"{key:>18}: {value}")

   negative_values: 0
       null_values: 0
               min: 77
               max: 11,071


## 5. Age and sex distribution

The FOI release also provides NCR-level age and sex breakdowns, one table per year. These are
**descriptive only** — they are aggregated at the regional level and are not used as LGU-level
regression predictors.

In [6]:
rows = []
for year in YEARS:
    df = pd.read_csv(RAW / f"dengue_datasets__age_sex_{year}.csv")
    r = check_age_sex(df, year)
    r["year"] = year
    rows.append(r)

age_sex_results = pd.DataFrame(rows).set_index("year")
age_sex_results

,age_groups_complete,female,male,total,sums_to_total,matches_foi
year,,,,,,
2021,True,4613,5880,10493,True,True
2022,True,19816,23937,43753,True,True
2023,True,10950,12728,23678,True,True
2024,True,17134,20091,37225,True,True
2025,True,20853,24161,45014,True,True


## 6. PSA census reference data

Two official census anchors are used:

| Year | Source | Status |
|---|---|---|
| 2020 | PSA Census of Population and Housing | Official |
| 2024 | PSA Census of Population (POPCEN) | Official |

Intermediate years (2021–2023) are interpolated and 2025 is extrapolated. Those values are
**computed in notebook 02**, not loaded from file, so that the estimation method is reproducible.

In [7]:
pop_2020 = pd.read_csv(REF / "ncr_population_masterlist__2020.csv")
pop_2024 = pd.read_csv(REF / "ncr_population_masterlist__2024.csv")

census = check_census_totals(pop_2020, pop_2024)
pd.DataFrame(census).T.assign(
    match=lambda d: d["match"].map({True: "OK", False: "MISMATCH"})
)

,computed,reported,match
2020,13484462,13484462,OK
2024,14001751,14001751,OK


## Summary

All data quality checks pass. The same checks run as assertions in `tests/test_validation.py`:

```
pytest -v
```

**Next:** notebook 02 handles LGU name standardisation, population interpolation and
extrapolation, density recomputation, and the merge into the 85-row LGU-year panel.